In [10]:
import argparse
import os
import pickle
import sys
import typing

import pandas as pd
import torch
from Bio import SeqIO
from typing import List, Union, Optional, Callable, Sequence
from transformers import AutoTokenizer, AutoModelForMaskedLM

from transformers import AutoModelForCausalLM
from tokenizers import Tokenizer
import torch
import torch.nn.functional as F

import einops

In [11]:
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)  

# Hooking utilities
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    FactoredMatrix,
    ActivationCache,
)

In [12]:
device = "cuda"

In [108]:
# 35 M
model_name = "facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

<img src="./ESM2_architecture.png" width="1000"/>

In [134]:
hooked_esm_config = HookedTransformerConfig(
    n_layers=12,
    d_model=480,
    d_head=24, # 480 -> 480 = 24 * 20
    n_heads=20,
    d_mlp=1920,
    d_vocab=33,
    n_ctx=59,
    act_fn="gelu",
    normalization_type="LN",
    positional_embedding_type="rotary",
    attention_dir="bidirectional",
    post_embedding_ln=False,
    tokenizer_name=model_name
)

In [135]:
test = HookedTransformer(hooked_esm_config)

In [137]:
def rotary_embeddings(inv_freq, cfg, device=device):
    """
    https://github.com/huggingface/transformers/blob/main/src/transformers/models/esm/modeling_esm.py#L80
    """
    t = torch.arange(cfg.n_ctx).to(inv_freq.device)
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1).to(inv_freq.device)
    cos_cached = emb.cos()
    sin_cached = emb.sin()
    
    return cos_cached.to(device), sin_cached.to(device)

def get_hooked_state_dict(hf_esm_state_dict, cfg, device=device):
    """
    hugging face ESM state dict -> hooked transformer state dict
    """
    old_state_dict_keys = hf_esm_state_dict.keys()
    new_state_dict = {}

    old_to_new_weights = {
        "attention.self.query.weight":"attn.W_Q",
        "attention.self.key.weight":"attn.W_K",
        "attention.self.value.weight":"attn.W_V",
        "attention.output.dense.weight":"attn.W_O"
    }
    old_to_new_bias = {
        "attention.self.query.bias":"attn.b_Q",
        "attention.self.key.bias":"attn.b_K",
        "attention.self.value.bias":"attn.b_V",
        "attention.output.dense.bias":"attn.b_O"
    }
    old_to_new_mlp = {
        "intermediate.dense.weight":"mlp.W_in",
        "intermediate.dense.bias":"mlp.b_in",
        "output.dense.weight":"mlp.W_out",
        "output.dense.bias":"mlp.b_out",
    }
    old_to_new_ln = {
        "attention.LayerNorm.weight":"ln1.w",
        "attention.LayerNorm.bias":"ln1.b",
        "LayerNorm.weight":"ln2.w",
        "LayerNorm.bias":"ln2.b"
    }

    # embedding matrix
    new_state_dict["embed.W_E"] = hf_esm_state_dict["esm.embeddings.word_embeddings.weight"]

    # temp unembedding matrix weight/bias (probably will delete later)
    new_state_dict["unembed.W_U"] = einops.rearrange(hf_esm_state_dict["lm_head.decoder.weight"], "d_vocab d_head -> d_head d_vocab")
    new_state_dict["unembed.b_U"] = torch.zeros(cfg.d_vocab)

    # calculating rotary embeddings (all of them are the same)
    cos_cached, sin_cached = rotary_embeddings(hf_esm_state_dict["esm.encoder.layer.0.attention.self.rotary_embeddings.inv_freq"], cfg, device)
    
    for l in range(cfg.n_layers):
        l_keys = [x for x in old_state_dict_keys if f".{l}." in x]
        old_prefix = f"esm.encoder.layer.{l}"
        new_prefix = f"blocks.{l}"

        # attn ignore = -inf
        new_state_dict[f"{new_prefix}.attn.IGNORE"] = torch.tensor(-torch.inf).to(device)
        
        # bidirectional attention, so attention should be looking everywhere
        new_state_dict[f"{new_prefix}.attn.mask"] = torch.full((cfg.n_ctx, cfg.n_ctx), True)

        # rotary embeddings
        new_state_dict[f"{new_prefix}.attn.rotary_cos"] = cos_cached
        new_state_dict[f"{new_prefix}.attn.rotary_sin"] = sin_cached
        
        # weights
        for w in old_to_new_weights.keys():
            # weights are arranged [out_features, in_features] = [n_head * d_head, d_model]
            new_weight_name = old_to_new_weights[w]
            if "output" in w:
                new_state_dict[f"{new_prefix}.{new_weight_name}"] = einops.rearrange(hf_esm_state_dict[f"{old_prefix}.{w}"], "(n_head d_head) d_model -> n_head d_head d_model", n_head=cfg.n_heads)
            else:
                new_state_dict[f"{new_prefix}.{new_weight_name}"] = einops.rearrange(hf_esm_state_dict[f"{old_prefix}.{w}"], "(n_head d_head) d_model -> n_head d_model d_head", n_head=cfg.n_heads)
            
        #biases
        for b in old_to_new_bias.keys():
            new_bias_name = old_to_new_bias[b]
            if "output" in b:
                new_state_dict[f"{new_prefix}.{new_bias_name}"] = hf_esm_state_dict[f"{old_prefix}.{b}"]
            else:
                new_state_dict[f"{new_prefix}.{new_bias_name}"] = einops.rearrange(hf_esm_state_dict[f"{old_prefix}.{b}"], "(n_head d_head) -> n_head d_head", n_head=cfg.n_heads)
            
        # mlp 
        for m in old_to_new_mlp.keys():
            # mlp are arranged [out_features, in_features] = [d_mlp, d_model]
            new_mlp_name = old_to_new_mlp[m]
            # mlp weights
            if "weight" in m:
                new_state_dict[f"{new_prefix}.{new_mlp_name}"] = einops.rearrange(hf_esm_state_dict[f"{old_prefix}.{m}"], "out_feats in_feats -> in_feats out_feats")
            # mlp biases
            else:
                new_state_dict[f"{new_prefix}.{new_mlp_name}"] = hf_esm_state_dict[f"{old_prefix}.{m}"]

        # layernorms
        for ln in old_to_new_ln.keys():
            new_ln_name = old_to_new_ln[ln]
            new_state_dict[f"{new_prefix}.{new_ln_name}"] = hf_esm_state_dict[f"{old_prefix}.{ln}"]

        # Final LayerNorm
        new_state_dict["ln_final.w"] = hf_esm_state_dict["esm.encoder.emb_layer_norm_after.weight"]
        new_state_dict["ln_final.b"] = hf_esm_state_dict["esm.encoder.emb_layer_norm_after.bias"]

    return new_state_dict

In [140]:
test.load_state_dict(get_hooked_state_dict(model.state_dict(), hooked_esm_config))

<All keys matched successfully>